In [1]:
import os

from google import genai
from google.genai import types
from google.genai import errors

# Set GEMINI_API_KEY and STOCK_TICKER in the environment before running this notebook.
symbol = os.environ.get("STOCK_TICKER", "").strip().upper()
if not symbol:
    raise RuntimeError("Set STOCK_TICKER before running enterprise.ipynb")

api_key = os.getenv("GEMINI_API_KEY", "").strip()
if not api_key:
    raise RuntimeError("Set GEMINI_API_KEY before running enterprise.ipynb")

client = genai.Client(api_key=api_key)

# Ordered list of models for fallback to handle rate limits
fallback_models = [
    "gemini-3.8-flash",
    "gemini-3.0-flash",       # Standardized "Gemini 3 Flash"
    "gemini-2.5-flash",
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-2.5-flash-lite"
]

description_response = None
model_errors = []

for model_name in fallback_models:
    try:
        description_response = client.models.generate_content(
            model=model_name,
            contents=(
                f"Describe the company identified by stock ticker {symbol}. "
                "If the ticker is unknown or ambiguous, say so instead of guessing."
            ),
            config=types.GenerateContentConfig(
                system_instruction="Answer professionally in no more than 200 words. Describe the company's business concisely, without bullet points. You may include historical facts.",
                temperature=0.2,
            ),
        )
        break
        
    except errors.APIError as e:
        model_errors.append(f"{model_name}: {e}")
        continue
    except Exception as e:
        model_errors.append(f"{model_name}: {e}")
        continue

# Check if we exhausted all models without getting a response
if description_response is None:
    error_details = "\n".join(model_errors)
    raise RuntimeError(
        f"Failed to retrieve a description for {symbol} after exhausting all fallback models.\n"
        f"Model errors:\n{error_details}"
    )

description = (description_response.text or "").strip()
if not description:
    raise RuntimeError(f"Gemini returned an empty description for {symbol}")

print(description)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Exxon Mobil Corporation, trading under the ticker XOM, is a multinational oil and gas corporation headquartered in Spring, Texas, and is one of the largest energy companies in the world. The company’s primary operations encompass the exploration, production, transportation, and sale of crude oil and natural gas. Additionally, ExxonMobil is a major manufacturer and marketer of petrochemicals, power generation products, and specialized commodities, while increasingly investing in lower-emission technologies such as carbon capture and biofuels. 

The company’s corporate lineage traces back to John D. Rockefeller’s Standard Oil trust, founded in 1870. Following the landmark 1911 antitrust dissolution of Standard Oil, the business split into several entities, most notably the Standard Oil Company of New Jersey (later known as Exxon) and the Standard Oil Company of New York (later known as Mobil). These two foundational pillars officially reunited in a historic merger in 1999 to form ExxonMo